# Make It Yours

The defaults work, but sometimes you need more. mlw has four customization hooks that let you plug in your own components without leaving the framework.

**What you'll learn:**
- `backend=` — bring your own sklearn-compatible algorithm
- `metrics=` — add custom scoring functions
- `splitter=` — use a custom cross-validation strategy
- `preprocessor=` — transform features before training
- `predict(proba=True)` — get probabilities instead of labels
- `embed()` — turn text into numbers
- `stack()` — ensemble multiple models

**Dataset:** Fraud detection (10,000 transactions, 2% fraud rate)

In [1]:
import ml
import numpy as np

## 1. Baseline

Start with the default workflow — this is our benchmark to beat.

In [2]:
data = ml.dataset("fraud")
ml.profile(data, "fraud")

── Profile [classification] ────────────
Rows:     10,000
Columns:  6
Target:   fraud
Balance:  2% minority class

  ! Imbalanced target: minority class 'yes' is 2% of data
  ! 296 rows (3.0%) have missing values in 'distance_km'



In [3]:
s = ml.split(data, "fraud", seed=42)
model = ml.fit(s.train, "fraud", seed=42)
ml.evaluate(model, s.valid)

── Metrics [classification] ────────────
accuracy:   0.9930
f1:         0.8158
precision:  0.8611
recall:     0.7750
roc_auc:    0.9915
(0.03s)

With only 2% fraud, the default XGBoost catches 78% of fraud cases (recall=0.775) with 86% precision. That's our bar.

## 2. Probabilities — `predict(proba=True)`

Labels ("fraud" / "not fraud") are crisp, but sometimes you want the model's *confidence*. Pass `proba=True` to get a probability for each class.

In [4]:
probs = ml.predict(model, s.valid, proba=True)
probs.head()

,no,yes
0,0.999721,0.000279
1,0.999966,0.000034
2,0.999972,0.000028
3,0.999994,0.000006
4,0.999994,0.000006


Each row sums to 1.0. The `yes` column is the fraud probability. These first 5 transactions are clearly not fraud (probability < 0.03%). 

Probabilities let you set custom thresholds: flag everything above 10% for manual review, or only auto-block above 90%.

## 3. Custom Metrics — `metrics=`

The built-in metrics (accuracy, F1, precision, recall, roc_auc) cover the basics. But you might want something specific — balanced accuracy, Matthews Correlation Coefficient, or a custom business metric.

Pass a dict of `{name: callable}` where each callable takes `(y_true, y_pred)` and returns a float.

In [5]:
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef

ml.evaluate(model, s.valid, metrics={
    "balanced_accuracy": balanced_accuracy_score,
    "mcc": matthews_corrcoef,
})

── Metrics [classification] ────────────
accuracy   0.9930   roc_auc            0.9915
f1         0.8158   balanced_accuracy  0.8862
precision  0.8611   mcc                0.8134
recall     0.7750
(0.03s)

Custom metrics appear alongside the built-in ones. **balanced_accuracy** (0.89) accounts for class imbalance — a fairer score than raw accuracy. **MCC** (0.81) is particularly good for imbalanced binary problems.

**Limitation:** Custom metrics receive hard predictions (labels), not probabilities. This means probability-based metrics like `log_loss` or `brier_score_loss` can't be passed directly here. For those, use `ml.predict(model, data, proba=True)` and compute them yourself.

## 4. Custom Backend — `backend=`

The 11 built-in algorithms cover most cases. But if you need something specific — a GradientBoostingClassifier, an IsolationForest, a PyTorch model — pass any sklearn-compatible estimator via `backend=`.

Requirements: must have `.fit(X, y)` and `.predict(X)` methods.

In [6]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model = ml.fit(s.train, "fraud", backend=gb, seed=42)
print(gb_model)

ml.evaluate(gb_model, s.valid)

── Model [GradientBoostingClassifier · classification] ─
Target:   fraud
Features: 5
Rows:     6,000  (0.3s)
Seed:     42
Hash:     25a7e38f



── Metrics [classification] ────────────
accuracy:   0.9920
f1:         0.7838
precision:  0.8529
recall:     0.7250
roc_auc:    0.9952
(0.02s)

The algorithm name shows `GradientBoostingClassifier` — whatever you pass in, the framework wraps it with the same splitting, evaluation, and serialization pipeline. You get all 15 verbs working with your custom model.

## 5. Custom Preprocessor — `preprocessor=`

Sometimes you need feature engineering before training: log transforms, interaction terms, domain-specific calculations. Pass a function that takes a DataFrame and returns a DataFrame.

In [7]:
def add_log_amount(X):
    """Log-transform the transaction amount (reduces skew)."""
    X = X.copy()
    if "amount" in X.columns:
        X["log_amount"] = np.log1p(X["amount"])
    return X

prep_model = ml.fit(s.train, "fraud", preprocessor=add_log_amount, seed=42)
print(prep_model)

ml.evaluate(prep_model, s.valid)

── Model [xgboost · classification] ────
Target:   fraud
Features: 5
Rows:     6,000  (0.02s)
Seed:     42
Hash:     da56a2f4



── Metrics [classification] ────────────
accuracy:   0.9930
f1:         0.8158
precision:  0.8611
recall:     0.7750
roc_auc:    0.9915
(0.03s)

The preprocessor runs automatically during training *and* prediction — no need to remember to apply it separately. Note: the preprocessor must be stateless (no fitting) so it can be applied to new data at prediction time.

In this case, the log transform didn't help (XGBoost handles skewed features well internally). But for linear models or when creating domain features, preprocessing can be the difference.

**Save/load caveat:** Models with custom preprocessors can be saved, but `ml.load()` will reject the custom function type as untrusted (security by design). For production deployment, wrap your preprocessor in a named class in a module that both the training and serving code can import.

## 6. Custom Splitter — `splitter=`

The default split is random stratified 60/20/20. But for time-series data, random splits are wrong — you'd be training on future data to predict the past.

Pass any sklearn-compatible CV splitter.

In [8]:
from sklearn.model_selection import TimeSeriesSplit

cv = ml.split(data, "fraud", seed=42, folds=3, splitter=TimeSeriesSplit(n_splits=3))
print(cv)

CVResult(folds=3, rows=10000)


With `TimeSeriesSplit`, each fold trains on all data before the split point and validates on the next chunk. This prevents future-leaking in time-ordered data.

Now fit a model using this CV strategy — pass the CVResult directly to `fit()`:

In [ ]:
ts_model = ml.fit(cv, "fraud", seed=42)
print(ts_model)

## 7. Stack — Ensemble Models

Different algorithms make different mistakes. `stack()` trains multiple base models and a "meta" model that learns to combine their predictions.

In [9]:
stacked = ml.stack(s.train, "fraud", 
                   models=["xgboost", "random_forest"], 
                   meta="logistic", seed=42)
print(stacked)

ml.evaluate(stacked, s.valid)

── Model [stacked · classification] ────
Target:   fraud
Features: 5
Rows:     6,000
Seed:     42
Hash:     3d867d0f



── Metrics [classification] ────────────
accuracy:   0.9950
f1:         0.8649
precision:  0.9412
recall:     0.8000
roc_auc:    0.9954
(0.05s)

The stacked model beats the solo XGBoost:
- F1: 0.82 &rarr; 0.86 (+5%)
- Precision: 0.86 &rarr; 0.94 (+8%)
- Recall: 0.78 &rarr; 0.80 (+3%)

Stacking works because XGBoost and Random Forest err differently — the logistic meta-learner figures out when to trust each one.

## 8. Compare — Which Approach Won?

In [10]:
ml.compare([model, gb_model, prep_model, stacked], s.valid)

── Compare [classification · 4 models] ─
 #                  algorithm  accuracy     f1  precision  recall  roc_auc  time
 4                    stacked     0.995 0.8649     0.9412   0.800   0.9954  0.04
 2 GradientBoostingClassifier     0.992 0.7838     0.8529   0.725   0.9952  0.03
 1                    xgboost     0.993 0.8158     0.8611   0.775   0.9915  0.03
 3                    xgboost     0.993 0.8158     0.8611   0.775   0.9915  0.03

The stacked ensemble (#4) wins on F1 and precision. The GradientBoosting backend (#2) has higher roc_auc but lower recall. The preprocessor (#3) didn't help this time — XGBoost doesn't need log transforms.

Fair comparison: all models evaluated on the same 2,000 validation transactions, no re-fitting.

## 9. Embed — Text Features

When your data has text columns, `embed()` turns them into numeric features you can feed into any model.

In [11]:
import pandas as pd

reviews = pd.Series([
    "Great product, love it!",
    "Terrible quality, broke immediately",
    "Decent value for the price",
    "Amazing service, highly recommend",
    "Worst purchase ever, total waste",
])

emb = ml.embed(reviews)
print(f"Vocab size: {emb.vocab_size} terms")
print(f"Vectors shape: {emb.vectors.shape}")

Vocab size: 22 terms
Vectors shape: (5, 22)


5 reviews become a 5x22 numeric matrix (TF-IDF features). The `Embedder` object stores the vocabulary so you can `emb.transform(new_texts)` at prediction time — same features, same columns.

Combine with `pd.concat([numeric_features, emb.vectors], axis=1)` to mix text and numeric data.

---

## Recap: The Four Hooks

| Hook | What it does | When to use |
|------|-------------|-------------|
| `backend=` | Custom sklearn-compatible estimator | Need an algorithm mlw doesn't ship |
| `metrics=` | Custom `(y_true, y_pred) → float` | Need business-specific scoring |
| `splitter=` | Custom CV strategy | Time-series or grouped data |
| `preprocessor=` | Custom `DataFrame → DataFrame` | Feature engineering |

Plus three bonus tools:
- `predict(proba=True)` — probabilities for custom thresholds
- `stack()` — ensemble multiple algorithms
- `embed()` — text to numbers

Every hook plugs into the same framework. You get `evaluate()`, `assess()`, `explain()`, `save()`, `compare()` — all working with your custom components.

**Good to know:** Each hook works independently. They don't compose with each other — for example, `stack()` doesn't accept `preprocessor=`, and you can't put `backend=` estimators inside a stack. If you need both custom preprocessing and stacking, preprocess your DataFrame before passing it to `stack()`.